# 04.04 离线测试、真机推理与章节实践

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 04.03 训练（或有课程提供的预训练模型）</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">对训练后的模型做离线测试，理解真机推理流程，完成章节实践</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">离线测试（实操）→ 真机推理（理论）→ 章节实践</td></tr>
</table>

## 第一部分：离线测试（实操）

训练完成后，我们在数据集上做**离线测试**——把数据集中的观察图像喂给模型，对比模型预测的动作和真实动作。这能快速评估模型是否学到了合理的动作映射，**不需要机械臂硬件**。

### 加载训练好的模型

可以用自己训练的模型，或课程提供的预训练模型：


---

### 部署到香橙派（昇腾 310 NPU）

训练好的 ACT 模型还可以部署到**香橙派 OrangePi AIPro**（搭载昇腾 Ascend 310B NPU），这是对华为昇腾产品的实际应用。本课程提供了适配昇腾 NPU 的推理脚本：

- **脚本位置**：`./src/run_inference_ascend.py`
- **硬件**：香橙派 AIPro + SO-101 机械臂 + 前置/腕部摄像头
- **关键适配**：通过 `torch_npu` 让 PyTorch 识别昇腾 NPU，模型 `.to("npu")` 加速推理

**香橙派环境要求**（详见脚本文件头注释）：
- CANN 8.0.RC1+（香橙派预装）
- PyTorch 2.1+ + torch_npu
- LeRobot + ffmpeg

**使用方法**（在香橙派上执行）：

```bash
python src/run_inference_ascend.py \
    --policy.path=./pretrained_model \
    --robot.port=/dev/ttyACM0 \
    --num_episodes=10 \
    --task="抓取方块"
```

> 💡 昇腾 310B 对 ACT 这类轻量模型（80M 参数）完全够用，实测可达 15-30 FPS。


In [ ]:
# ===== 加载数据集（独立运行时需要）=====
from lerobot.datasets.lerobot_dataset import LeRobotDataset
dataset = LeRobotDataset(repo_id='local/so101_block', root='./src/data_final', video_backend='pyav')
print(f'数据集: {dataset.meta.total_episodes} episodes')

# ===== 加载训练好的 ACT 模型 =====
import os, torch
from lerobot.policies.act.modeling_act import ACTPolicy

MODEL_MS_ID = 'Kumako/so101_act_pretrained'  # ModelScope 预训练模型 ID

# 候选模型路径（按优先级）：
# 1. 自己完整训练的模型  2. 自己快速测试的模型  3. 课程提供的预训练模型
model_paths = [
    "outputs/train/act_so101/checkpoints/last",      # 自己完整训练
    "outputs/train/act_so101_test/checkpoints/last",  # 自己快速测试
    "./src/pretrained_model",                          # 课程预训练模型
]
model_path = None
for p in model_paths:
    if os.path.exists(os.path.join(p, "config.json")):
        model_path = p
        print(f"✅ 找到模型: {p}")
        break

# 若本地都没有，自动从 ModelScope 下载课程预训练模型
if model_path is None:
    print(f"⏳ 未找到本地模型，自动下载课程预训练模型...")
    print(f"   模型 ID: {MODEL_MS_ID}（约 197MB，首次下载 2-5 分钟）")
    try:
        import modelscope
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', 'modelscope',
                             '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple', '-q'])
    from modelscope import snapshot_download
    snapshot_download(MODEL_MS_ID, local_dir="./src/pretrained_model")
    model_path = "./src/pretrained_model"
    print(f"✅ 下载完成: {model_path}")

# 加载模型（先清理 config.json 里 v0.4.2 不认识的新字段，避免 DecodingError）
import json as _json
cfg_path = os.path.join(model_path, "config.json")
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        cfg = _json.load(f)
    # v0.4.2 不认识的新版字段，删掉避免 DecodingError
    _new_fields = ['use_peft', 'temporal_ensemble_coeff']
    _removed = [k for k in _new_fields if k in cfg]
    if _removed:
        for k in _removed:
            del cfg[k]
        with open(cfg_path, 'w') as f:
            _json.dump(cfg, f, indent=2)
        print(f"💡 已清理 config.json 的不兼容字段: {_removed}")

policy = ACTPolicy.from_pretrained(model_path)
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    import torch_npu  # noqa: F401
    if torch.npu.is_available():
        _dev = 'npu'
except Exception:
    pass
policy.eval()
policy.to(_dev)
print(f"✅ 模型已加载到 {_dev}")

### 对单 episode 做离线推理

取出数据集中的一个 episode，逐帧用模型预测动作，对比真实动作：

In [ ]:
# ===== 离线推理：对比预测动作与真实动作 =====
import matplotlib.pyplot as plt
import numpy as np

if model_path:
    # 取一个 episode 的前 N 帧
    N = 100  # 测试帧数
    start = 0
    real_actions = []
    pred_actions = []

    # 用 Cell[1] 定义的设备 _dev（v0.4.2 没有 policy.device 属性）
    infer_device = _dev if '_dev' in dir() else ('npu' if torch.npu.is_available() else 'cpu')
    print(f"推理设备: {infer_device}")

    with torch.no_grad():
        for i in range(start, min(start + N, len(dataset))):
            sample = dataset[i]
            # 真实动作
            real_action = sample['action'].numpy() if hasattr(sample['action'], 'numpy') else np.array(sample['action'])
            real_actions.append(real_action)

            # 模型预测（需要构造 batch）
            # 只保留 observation.* 键喂给 policy（不含 action 等标签）
            batch = {k: v.unsqueeze(0).to(infer_device)
                     for k, v in sample.items()
                     if k.startswith('observation.') and hasattr(v, 'unsqueeze')}
            try:
                pred = policy.select_action(batch)
                pred_actions.append(pred.cpu().numpy().flatten()[:6])
            except Exception as e:
                print(f"  ⚠️ Frame {i} inference failed: {e}")
                pred_actions.append(real_action)

    real_actions = np.array(real_actions)
    pred_actions = np.array(pred_actions)

    # 可视化前 3 个关节的对比
    action_names = ['shoulder_pan', 'shoulder_lift', 'elbow_flex']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, (ax, name) in enumerate(zip(axes, action_names)):
        ax.plot(real_actions[:, i], label='Real Action', color='steelblue', linewidth=2)
        ax.plot(pred_actions[:, i], label='Predicted', color='coral', linewidth=2, linestyle='--')
        ax.set_title(f'Joint {i}: {name}')
        ax.set_xlabel('Frame'); ax.set_ylabel('Action Value')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle('Offline Test: Predicted vs Real Action (first 3 joints)', fontsize=13)
    plt.tight_layout()
    plt.show()
    print("💡 The closer the predicted (dashed orange) matches the real (solid blue), the better the model.")
else:
    print("⚠️ No model available, skipping offline test. Please load a model first.")

### 离线测试的意义

离线测试用训练数据评估，**只能判断模型是否"记住"了训练数据**，不能反映真实泛化能力。真正的评估需要在机械臂上做**真机推理**（见下方理论部分）。

---

## 第二部分：真机推理理论（需机械臂硬件）

> ⚠️ 本部分为理论讲解。真机推理需要 SO-101 机械臂 + 摄像头，**云环境无法执行**。

### 真机推理流程

LeRobot 用 `lerobot-record` 命令做真机推理（也叫"记录评估 episode"）：

```text
启动 Follower 臂 + 摄像头 → 加载训练好的策略 → 循环{拍摄画面 → 策略预测动作 → 执行动作}
```

### lerobot-record 命令

```bash
lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_robot \
    --robot.cameras='{front: {type: opencv, index_or_path: 0, width: 640, height: 360, fps: 30},
                       wrist: {type: opencv, index_or_path: 1, width: 640, height: 360, fps: 30}}' \
    --display_data=true \
    --dataset.repo_id=local/eval_act_so101 \
    --dataset.num_episodes=10 \
    --dataset.single_task="抓取方块" \
    --policy.path=outputs/train/act_so101/checkpoints/last
```

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">参数</th><th align="left">含义</th></tr>
<tr><td align="left"><code>--robot.type</code></td><td align="left">机器人型号（so101_follower 表示 SO-101 从臂）</td></tr>
<tr><td align="left"><code>--robot.port</code></td><td align="left">机械臂连接的串口（Linux 通常 /dev/ttyACM0）</td></tr>
<tr><td align="left"><code>--robot.cameras</code></td><td align="left">摄像头配置（前置 + 腕部）</td></tr>
<tr><td align="left"><code>--policy.path</code></td><td align="left">训练好的模型路径</td></tr>
<tr><td align="left"><code>--dataset.num_episodes</code></td><td align="left">评估多少次（如 10 次，统计成功率）</td></tr>
</table>

### 评估指标：成功率

真机评估的核心指标是**任务成功率**：在 N 次评估中，机械臂成功完成任务的次数占比。例如 10 次评估成功 7 次，成功率 70%。

---

## 第三部分：章节实践

本章实践包含 **2 道** 综合题。

### 综合实践题 1：用预训练模型对自定义数据推理

**任务**：使用课程提供的预训练模型（`pretrained_model.zip`），完成以下操作：
1. 解压 `pretrained_model.zip` 到 `./pretrained_model`；
2. 加载预训练的 ACT 策略；
3. 从数据集中取 5 个不同 episode 的首帧，分别推理；
4. 对比 5 次预测动作的差异，思考：为什么同一个任务的不同起点，预测动作不同？

请在下方 code cell 完成：

In [ ]:
# ===== 综合实践题 1：你的代码 =====
# 任务：加载预训练模型，对 5 个 episode 的首帧做推理，保存可视化结果
#
# 提示：
# - 预训练模型已自动下载到 ./src/pretrained_model（见上方 Cell[1]）
# - 用 dataset[i] 取样本，policy.predict(sample) 做推理
#
# import torch, numpy as np
# import matplotlib.pyplot as plt
#
# # Step 1: 模型已加载（policy），数据集已加载（dataset）
#
# # Step 2: 取 5 个 episode 的首帧推理
# episode_starts = [0, 681, 1362, 2043, 2724]
# fig, axes = plt.subplots(1, 5, figsize=(20, 4))
# for ax, ep_start in zip(axes, episode_starts):
#     sample = dataset[ep_start]
#     batch = {k: v.unsqueeze(0).to(policy.device) for k, v in sample.items() if hasattr(v, 'unsqueeze')}
#     with torch.inference_mode():
#         action = policy.select_action(batch)
#     ax.imshow(sample['observation.images.front'].permute(1, 2, 0).numpy())
#     ax.set_title(f"ep@{ep_start}")
#     ax.axis('off')
# plt.tight_layout()
# plt.show()
print("💡 请取消上方注释，补全代码后运行")

### 综合实践题 2：训练超参对比设计（思考题）

**任务**：设计一个实验，对比不同训练配置对 ACT 性能的影响。回答：

(a) 如果只有 30 分钟训练时间，你会如何设置 `--steps`、`--batch_size`、`--save_freq`？

(b) 如果训练后发现机械臂动作抖动严重，可能是什么原因？如何用 ACT 的参数改善？

(c) 为什么离线测试 loss 很低，但真机成功率却不高？（至少列 1 个原因）

请在下方作答：

In [ ]:
# ===== 综合实践题 2：你的回答 =====
print("(a) 30分钟训练配置:______")
print("(b) 动作抖动原因+改善:______")
print("(c) 离线好但真机差的原因:______")


---

## 参考答案

In [ ]:
# 查看答案
!cat ./answer/04.04_eval_practice/practice1.txt
print("\n" + "="*50 + "\n")
!cat ./answer/04.04_eval_practice/practice2.txt
